# Demostración y Solución del Problema de Mínimos Cuadrados Ordinarios (MCO)

*Universidad Militar Nueva Granada — Matemáticas Aplicadas y Computacionales*

---

## 1. Planteamiento del Problema

Dado un conjunto de $m$ observaciones empíricas y $n$ variables explicativas (con $m > n$), el modelo lineal se formula matricialmente como el sistema **sobredeterminado**:

$$
A x = b \tag{1}
$$

donde:

- $A \in \mathbb{R}^{m \times n}$ representa la matriz de diseño.
- $x \in \mathbb{R}^{n}$ es el vector de parámetros a estimar.
- $b \in \mathbb{R}^{m}$ es el vector de observaciones de la variable de respuesta.

Ante la imposibilidad de hallar una solución exacta (ya que $b \notin C(A)$, el espacio columna de $A$), el método de mínimos cuadrados busca **proyectar ortogonalmente** $b$ sobre $C(A)$. Se define la estimación del vector observacional como $\hat{b} = A\hat{x} \in C(A)$, reduciendo así la distancia entre el valor real y el subespacio proyectado.

---

## 2. Estructura Formal y Solución Analítica

### Definición 1 (Vector de Error)

El vector de residuos $e \in \mathbb{R}^{m}$ correspondiente al error de ajuste se define formalmente como:

$$
e = b - \hat{b} = b - A\hat{x} \tag{2}
$$

### Axioma 1 (Ecuaciones Normales)

Para que el residuo $e$ sea ortogonal a la imagen de $A$, el estimador óptimo debe satisfacer de manera fundamental las **ecuaciones normales**:

$$
A^{T} A \hat{x} = A^{T} b \tag{3}
$$

### Proposición 1 (Deducción del Estimador Óptimo)

Bajo la condición de que la matriz de diseño tenga rango columna completo, es decir, $\text{rango}(A) = n$, la matriz **Gramiana** $(A^{T} A) \in \mathbb{R}^{n \times n}$ es estrictamente invertible.

**Desarrollo analítico.** Multiplicando ambos lados de las ecuaciones normales por la inversa $(A^{T} A)^{-1}$, aislamos directamente el vector de parámetros:

$$
(A^{T} A)^{-1} (A^{T} A) \hat{x} = (A^{T} A)^{-1} A^{T} b \tag{4}
$$

$$
I_n \, \hat{x} = (A^{T} A)^{-1} A^{T} b \tag{5}
$$

Obteniendo el **estimador analítico general de mínimos cuadrados**:

$$
\boxed{\hat{x} = (A^{T} A)^{-1} A^{T} b} \tag{6}
$$

---

## 3. Aplicación: Caso de Regresión Lineal Simple

Para el modelo de una recta con ecuación $y = mx + b$, adaptamos los coeficientes ordenando respecto a la pendiente ($m$) y el intercepto ($b$):

$$
y_i = m \cdot x_i + b \cdot 1 \tag{7}
$$

Definimos la estructuración matricial del modelo:

**Vector de coeficientes desconocidos:**

$$
\hat{x} = \begin{bmatrix} m \\ b \end{bmatrix}
$$

**Matriz de diseño para $N$ observaciones:**

$$
A = \begin{bmatrix} x_1 & 1 \\ x_2 & 1 \\ \vdots & \vdots \\ x_N & 1 \end{bmatrix} \tag{8}
$$

$$
A^{T} A = \begin{bmatrix} x_1 & x_2 & \cdots & x_N \\ 1 & 1 & \cdots & 1 \end{bmatrix} \begin{bmatrix} x_1 & 1 \\ x_2 & 1 \\ \vdots & \vdots \\ x_N & 1 \end{bmatrix} = \begin{bmatrix} \sum x_i^{2} & \sum x_i \\ \sum x_i & N \end{bmatrix} \tag{9}
$$

El producto matricial fundamental $A^{T} A$ resulta en una matriz **simétrica de dimensiones $2 \times 2$**. Sustituyendo en la expresión óptima, los parámetros se obtienen evaluando:

$$
\begin{bmatrix} m \\ b \end{bmatrix} = \left( \begin{bmatrix} \sum x_i^{2} & \sum x_i \\ \sum x_i & N \end{bmatrix} \right)^{-1} A^{T} y \tag{10}
$$

---

## 4. Equivalencia Computacional en Python

La fundamentación algebraica anterior es exactamente la que ejecuta el entorno numérico de **NumPy**.

### 4.1. Construcción de la Matriz de Diseño

```python
A_lin = np.vstack([x, np.ones(len(x))]).T
```

- `np.ones(len(x))`: crea el vector de unos para asociarlo al intercepto $b$.
- `np.vstack([...])`: une el vector $x$ y el de unos en dos filas ($2 \times N$).
- `.T`: transpone la matriz resultante a dimensión $N \times 2$, correspondiente a $A$.

### 4.2. Resolución del Sistema

```python
m_lin, c_lin = np.linalg.lstsq(A_lin, y, rcond=None)[0]
```

La función `np.linalg.lstsq` resuelve internamente las ecuaciones normales mediante una **descomposición de rango completo** para hallar $\hat{x} = (A^{T} A)^{-1} A^{T} y$.

### 4.3. Proyección Ortogonal de la Respuesta

```python
ajuste_lin = m_lin * x + c_lin
```

En álgebra lineal, este paso equivale formalmente a computar la proyección de estimación:

$$
\hat{y} = A \hat{x} \tag{11}
$$

obteniendo el **vector de respuesta proyectado** sobre el subespacio del modelo.

In [3]:
import pandas as pd
import numpy as np

# Cargar los datos (asumiendo que ya pasaron por tu función de limpieza en el Notebook 1)
df = pd.read_csv('../data/processed/datos_limpios.csv', sep=';')
x = df['edad'].values
y = df['tiempo_en_calle_meses'].values

# ==========================================
# AJUSTES POR MÍNIMOS CUADRADOS (np.linalg.lstsq)
# ==========================================

# 1. Modelo Lineal: y = mx + c
A_lin = np.vstack([x, np.ones(len(x))]).T
m_lin, c_lin = np.linalg.lstsq(A_lin, y, rcond=None)[0]
ajuste_lin = m_lin * x + c_lin
print(f"Modelo Lineal: y = {m_lin:.4f}x + {c_lin:.4f}")


# 2. Modelo Cuadrático: y = ax^2 + bx + c
A_cuad = np.vstack([x**2, x, np.ones(len(x))]).T
a_cuad, b_cuad, c_cuad = np.linalg.lstsq(A_cuad, y, rcond=None)[0]
ajuste_cuad = a_cuad * x**2 + b_cuad * x + c_cuad
print(f"Modelo Cuadrático: y = {a_cuad:.4f}x^2 + {b_cuad:.4f}x + {c_cuad:.4f}")

# 3. Modelo Cúbico: y = ax^3 + bx^2 + cx + d
A_cub = np.vstack([x**3, x**2, x, np.ones(len(x))]).T
a_cub, b_cub, c_cub, d_cub = np.linalg.lstsq(A_cub, y, rcond=None)[0]
ajuste_cub = a_cub * x**3 + b_cub * x**2 + c_cub * x + d_cub
print(f"Modelo Cúbico: y = {a_cub:.4f}x^3 + {b_cub:.4f}x^2 + {c_cub:.4f}x + {d_cub:.4f}")

# 4. Modelo Exponencial: ln(y) = mx + ln(c)  =>  y = c * e^(mx)
A_exp = np.vstack([x, np.ones(len(x))]).T
m_exp, ln_c_exp = np.linalg.lstsq(A_exp, np.log(y), rcond=None)[0]
c_exp = np.exp(ln_c_exp)
ajuste_exp = c_exp * np.exp(m_exp * x)
print(f"Modelo Exponencial: y = {c_exp:.4f} * e^({m_exp:.4f}x)")

# 5. Modelo Logarítmico: y = m*ln(x) + c
A_log = np.vstack([np.log(x), np.ones(len(x))]).T
m_log, c_log = np.linalg.lstsq(A_log, y, rcond=None)[0]
ajuste_log = m_log * np.log(x) + c_log
print(f"Modelo Logarítmico: y = {m_log:.4f} * ln(x) + {c_log:.4f}")
# ==========================================
# EXPORTAR RESULTADOS
# ==========================================
df_ajustes = pd.DataFrame({
    'edad': x,
    'tiempo_real': y,
    'ajuste_lineal': ajuste_lin,
    'ajuste_cuadratico': ajuste_cuad,
    'ajuste_cubico': ajuste_cub,
    'ajuste_exponencial': ajuste_exp,
    'ajuste_logaritmico': ajuste_log
})

# Ordenar por 'x' es crucial para que las curvas en el Notebook 3 se dibujen sin rayones extraños
df_ajustes = df_ajustes.sort_values(by='edad')
df_ajustes.to_csv('../outputs/tablas/02_ajustes_completos.csv', index=False, sep=';')


Modelo Lineal: y = 4.6086x + -34.5050
Modelo Cuadrático: y = -0.0989x^2 + 13.5253x + -216.4969
Modelo Cúbico: y = -0.0033x^3 + 0.3628x^2 + -6.2972x + 46.8364
Modelo Exponencial: y = 21.1382 * e^(0.0340x)
Modelo Logarítmico: y = 196.3043 * ln(x) + -564.4795
